# A certified upper bound for the growth constant

This notebook implements the upper-bound calculation in Section IV-B of *Asymptotic Growth of the Number of Rubik's Snake Shapes*. It runs on its own and uses no unpublished files. It requires `../rubiks_snake.py`, NumPy, Numba, SciPy, the standard library, and a Jupyter runtime.

Vertices are all valid rotation words of length $m$. Every valid word of length $m+1$ gives an edge from its prefix to its suffix. We use the column convention $A_{v,u}=\#(u\to v)$. Every globally valid word gives a graph path, so $\mu\leq\rho(A_m)$.

Decomposing the graph into strongly connected components (SCCs) puts the matrix in block-triangular form. Its spectral radius is the maximum over diagonal blocks. We keep the internal edges of every cyclic component, including singleton self-loops, and discard only zero diagonal blocks. Iteration with $A+I$, normalized independently in each SCC, supplies a candidate vector. We round it to a positive integer vector and check $D A v\leq p v$ exactly, without overflow. This integer check certifies $\rho(A)\leq p/D$ independently of the floating-point iteration.

Outputs through $m=11$ were reproduced from these cell sources on 2026-09-19 (Python 3.14.6, NumPy 2.5.3, Numba 0.67.0, SciPy 1.18.1). The final $m=13$ output is the archived research result from 2026-09-11 and has not been rerun with this notebook. Execution counts are null because validation ran in a script outside the editor kernel. Rerunning regenerates all windows and verifies the certificates using overflow-free integer comparisons.

The $m=13$ run used about 195 seconds and 6.5 GiB; runtime and peak memory may differ on another run. Skip the final large cell if the machine has insufficient memory. The work grows exponentially with $m$. Although the enumerator accepts $m\leq18$, a run within this limit can still exhaust memory.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys
started = perf_counter()
cwd = Path.cwd()
candidates = (cwd, cwd / 'rubiks-snake', cwd.parent, cwd.parent / 'rubiks-snake')
snake_dir = next((p for p in candidates if (p / 'rubiks_snake.py').is_file()), None)
if snake_dir is None:
    raise FileNotFoundError('Run from asymptotic-analysis, rubiks-snake, or the repository root')
sys.path.insert(0, str(snake_dir.resolve()))
import numpy as np
import numba
import scipy
from rubiks_snake import window_upper_bound
print(f'Python {sys.version.split()[0]}; NumPy {np.__version__}; Numba {numba.__version__}; SciPy {scipy.__version__}')
print(f'Setup: {perf_counter() - started:.3f} s')

In [ ]:
def get_bound(m, iterations=500, scale=10**12, denominator=10**9):
    """Construct the window graph and return an exactly verified spectral bound."""
    started = perf_counter()
    result = window_upper_bound(m, iterations, scale, denominator)
    result['seconds'] = perf_counter() - started
    q = result['bound']
    print(f"m={m}: {result['states']} cyclic states, {result['edges']} internal edges")
    print(f"mu <= {q} = {float(q):.9f}; exact certificate passed; {result['seconds']:.3f} s")
    return result

## Small examples

Increase `m` to check collisions within a longer window. More iterations or a larger `scale` can improve the certificate vector; `denominator` controls the rational resolution. Every returned bound passes the exact check, regardless of these parameter choices.

In [ ]:
SMALL_WINDOWS = (3, 5, 7)
small_results = [get_bound(m) for m in SMALL_WINDOWS]

m=3: 59 cyclic states, 225 internal edges
mu <= 3810528899/1000000000 = 3.810528899; exact certificate passed; 0.024 s
m=5: 822 cyclic states, 3062 internal edges
mu <= 1860055621/500000000 = 3.720111242; exact certificate passed; 0.007 s
m=7: 11293 cyclic states, 41819 internal edges
mu <= 370307551/100000000 = 3.703075510; exact certificate passed; 0.074 s


## Published certificates

The first two runs use much less memory than the final calculation, which has its own cell. Edit `PUBLISHED_RUNS` or `LARGE_WINDOW` to change the computation; each tuple specifies `(m, iterations)`. The final run is the one that used about 6.5 GiB. State and edge counts cover only the cyclic diagonal blocks of the full graph. These certificates bound the growth rate and do not supply a pointwise exponential prefactor.

In [ ]:
PUBLISHED_RUNS = [(9, 500), (11, 500)]
published_results = [get_bound(m, iterations) for m, iterations in PUBLISHED_RUNS]

m=9: 153306 cyclic states, 565050 internal edges
mu <= 1842734183/500000000 = 3.685468366; exact certificate passed; 1.381 s
m=11: 2070446 cyclic states, 7608583 internal edges
mu <= 734968961/200000000 = 3.674844805; exact certificate passed; 26.251 s


### Largest published run: several GiB of memory

This cell rebuilds the graph and checks the certificate without reading the saved result. Skip it if insufficient memory is available. The saved output is archival and still needs a fresh run of the overflow-safe check.

In [ ]:
LARGE_WINDOW, LARGE_ITERATIONS = 13, 180
largest = get_bound(LARGE_WINDOW, LARGE_ITERATIONS)

Archived reference result (2026-09-11; original implementation):
m=13: 27847372 cyclic states, 102127233 internal edges; mu <= 3.667542939
Historical elapsed time: 194.79 s; peak memory: 6828692 kB
